Read out output metadata of ETL pipeline.

In [1]:
import pandas as pd
import config as c

In [2]:
dfs = []
for i in range(40):
    path = (str(c.PREPROCESSED_EEG_DIR) + f"/S09/chunk_{i:04d}/metadata.parquet")
    dfs.append(pd.read_parquet(path))

df = pd.concat(dfs, ignore_index=True)
print(df)

        exam_id                                   original_exam_id     sub_id  \
0    exam-00131  20220426-170148-{4d668cd5-9e5b-4ea5-8236-60c52...    sub-119   
1    exam-00132  20220426-165659-{61a7605b-aaaa-4a07-a455-847d2...    sub-120   
2    exam-00133  20220426-165657-{c5283e16-a39d-49a3-ae8e-4e2f0...    sub-121   
3    exam-00135  20220426-165650-{23f94ee0-d494-43fb-b85f-0bd67...    sub-123   
4    exam-00138  20220426-170148-{896f8a55-d1bd-4b0f-b18c-db383...    sub-126   
..          ...                                                ...        ...   
811  exam-35391  20220426-165604-{af9bec17-29fd-4c3c-95ee-b642a...  sub-29571   
812  exam-35392  20220426-165601-{6bebc359-714e-4d56-88e0-1e375...  sub-29572   
813  exam-35393  20220426-165601-{78f9e25f-995d-4877-bf0c-4fd49...  sub-29573   
814  exam-35399  20220426-165554-{6e275af5-021a-4dd8-95d5-956ab...  sub-29579   
815  exam-35401  20220426-165552-{ca44d3f2-5b90-4271-baf3-ac1be...  sub-29581   

     target site   age  sex

In [3]:
df.drop(["original_exam_id", "site", "orig_sfreq", "n_good_windows", "idx"], axis = 1)

,exam_id,sub_id,target,age,sex,chunk_idx,row_in_chunk
0,exam-00131,sub-119,0,49.8,0,0,0
1,exam-00132,sub-120,1,53.7,1,0,1
2,exam-00133,sub-121,1,49.8,1,0,2
3,exam-00135,sub-123,0,46.6,1,0,3
4,exam-00138,sub-126,1,29.8,1,0,4
...,...,...,...,...,...,...,...
811,exam-35391,sub-29571,0,26.6,1,39,4
812,exam-35392,sub-29572,0,40.8,1,39,5
813,exam-35393,sub-29573,0,49.9,0,39,6
814,exam-35399,sub-29579,0,34.1,0,39,7


## spot for longutudinal patients

In [4]:
df['sub_id'].value_counts()

sub_id
sub-10010    3
sub-26252    2
sub-16602    2
sub-6781     2
sub-22968    2
            ..
sub-9982     1
sub-9984     1
sub-9986     1
sub-9987     1
sub-9966     1
Name: count, Length: 801, dtype: int64

In [5]:
df[df['sub_id'] == 'sub-10010']

,exam_id,original_exam_id,sub_id,target,site,age,sex,idx,orig_sfreq,n_good_windows,chunk_idx,row_in_chunk
286,exam-11868,20220426-165709-{e58c4718-9530-4792-8b01-7ee18...,sub-10010,1,S09,19.2,1,11868,250.0,211,14,17
287,exam-11870,20220426-170130-{7816ec3f-5062-4315-8aa4-b087f...,sub-10010,1,S09,18.2,1,11870,250.0,205,14,18
288,exam-11871,20220426-165853-{6e6a35f1-0818-4537-afe5-4d2e8...,sub-10010,1,S09,18.6,1,11871,250.0,200,14,19


Need to be stratified not to cause data leakage!

In [6]:
df['target'].value_counts()

target
0    467
1    349
Name: count, dtype: int64

In [7]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=0)
df["fold"] = -1

for fold, (_, val_idx) in enumerate(
    sgkf.split(df, df["target"].astype(int), groups=df["sub_id"])):
    df.loc[val_idx, "fold"] = fold

In [8]:
df.groupby("fold")["target"].value_counts(), df.groupby("sub_id")["fold"].nunique().max()

(fold  target
 0     0          88
       1          79
 1     0          98
       1          63
 2     0         101
       1          61
 3     0          89
       1          75
 4     0          91
       1          71
 Name: count, dtype: int64,
 np.int64(1))